In [ ]:
# Edit these four values.
CHROMOSOME = 'chr17'

In [ ]:
WORKING_DIR= '/home/jupyter/fine_shards/small'

JOINT_VCF     = f"{WORKING_DIR}/PacBio-Exposomics2-{CHROMOSOME}.g.vcf.bgz"
JOINT_VCF_IDX = f"{WORKING_DIR}/PacBio-Exposomics2-{CHROMOSOME}.g.vcf.bgz.tbi"  # A .csi path is also fine.
OUTPUT_TSV    = f"{WORKING_DIR}/split_short_regions.{CHROMOSOME}.tsv"
TARGET_SITES  = 500_000

# Optional settings.
BCFTOOLS = "bcftools"  # Or an explicit path to the executable.
PROGRESS_EVERY = 1_000_000  # Set to 0 for no progress messages.
HEAD_ROWS = 10

The following contets are generated by ChatGPT.

# Shard a chromosome pVCF by record count

Set the local pVCF path, its index path, the target records per shard, and the output TSV in the next cell. Then **Run All**. Requires Python 3.10+, pandas, and `bcftools` on your Jupyter kernel's PATH. The notebook is self-contained.

The output has the eight requested columns, including `entity:split_short_regions_id`. Column 2 holds the calculated number of **VCF records**, column 3 is `TBD` followed by the four-digit shard ID, and column 6 is `TBD`. Multiple records at the same POS each count once. A multiallelic VCF record counts once. No records are filtered.

Intervals are **1-based and inclusive**. The first starts at base **1**; the final interval ends at the maximum variant end, rather than the contig length. Adjacent intervals are exactly contiguous.

In [ ]:
# Edit these four values.
CHROMOSOME = 'chrM'
WORKING_DIR= '/home/jupyter/fine_shards/small'

JOINT_VCF     = f"{WORKING_DIR}/PacBio-Exposomics2.{CHROMOSOME}.GLnexus.bcf"
JOINT_VCF_IDX = f"{WORKING_DIR}/PacBio-Exposomics2.{CHROMOSOME}.GLnexus.bcf.csi"  # A .csi path is also fine.
OUTPUT_TSV    = f"{WORKING_DIR}/split_short_regions.{CHROMOSOME}.tsv"
TARGET_SITES  = 500_000

# Optional settings.
BCFTOOLS = "bcftools"  # Or an explicit path to the executable.
PROGRESS_EVERY = 1_000_000  # Set to 0 for no progress messages.
HEAD_ROWS = 10

## Boundary calculation

A record's end is its scalar `INFO/END`, when present. Otherwise it is `POS + len(REF) - 1`. Records must be on one chromosome and sorted by POS. Missing `END` is represented by `.`. Malformed ends and decreasing positions stop the calculation.

The scan maintains the maximum end of the current overlapping group. A new group starts only when the next POS is greater than that maximum. This keeps shared positions, nested variants, and chains of overlapping variants together.

For each shard, the code compares the whole-group record counts immediately below and above the target and chooses the closer count. Equal-distance ties choose the smaller count. An indivisible group can exceed the target. The final shard holds the remaining records.

Each internal interval ends at `next_shard_first_POS - 1`; the next begins one base later. Because the previous group's maximum end is strictly less than that next POS, every record's entire `[POS, end]` span belongs to exactly one interval. Memory use grows with the number of shards, rather than the number of records.

In [ ]:
from contextlib import contextmanager
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
import os
import shlex
import shutil
import subprocess
import tempfile
import time
import warnings

import pandas as pd


COLUMNS = [
    "entity:split_short_regions_id",
    "expected_number_of_sites",
    "gcs_output_dir",
    "joint_vcf",
    "joint_vcf_idx",
    "number_of_sites",
    "output_tag",
    "region",
]


@dataclass(frozen=True)
class Span:
    """A whole overlap group, or several whole groups assigned to one shard."""
    chrom: str
    first_pos: int
    end: int
    sites: int


def positive_integer(value, name):
    if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
        raise ValueError(f"{name} must be a positive integer.")
    return value


def overlap_groups(lines: Iterable[str], stats: dict, progress_every: int = 0):
    """Stream CHROM, POS, REF, INFO/END rows into indivisible overlap groups."""
    previous_pos = 0
    chrom = None
    first_pos = max_end = group_sites = 0
    started = time.perf_counter()

    for record_number, line in enumerate(lines, 1):
        fields = line.rstrip("\r\n").split("\t")
        if len(fields) != 4:
            raise ValueError(f"Record {record_number}: expected four query fields.")
        current_chrom, pos_text, ref, end_text = fields
        if not current_chrom or current_chrom == ".":
            raise ValueError(f"Record {record_number}: missing chromosome.")
        try:
            pos = int(pos_text)
        except ValueError as exc:
            raise ValueError(f"Record {record_number}: invalid POS {pos_text!r}.") from exc
        if pos < 1:
            raise ValueError(f"Record {record_number}: POS must be at least 1.")

        if end_text == ".":
            if not ref or ref == ".":
                raise ValueError(f"Record {record_number}: missing REF and INFO/END.")
            end = pos + len(ref) - 1
        else:
            try:
                end = int(end_text)
            except ValueError as exc:
                raise ValueError(
                    f"Record {record_number}: INFO/END must be one integer or '.', "
                    f"got {end_text!r}."
                ) from exc
        if end < pos:
            raise ValueError(f"Record {record_number}: END {end} is before POS {pos}.")

        if chrom is None:
            chrom = current_chrom
            stats["chromosome"] = chrom
        elif current_chrom != chrom:
            raise ValueError(
                f"Expected one chromosome; found {current_chrom!r} after {chrom!r}."
            )
        if pos < previous_pos:
            raise ValueError(
                f"Record {record_number}: input is not POS-sorted ({pos} < {previous_pos})."
            )
        previous_pos = pos
        stats["records"] = record_number
        stats["maximum_end"] = max(stats["maximum_end"], end)

        # Inclusive spans overlap when the next POS is at or before max_end.
        # max_end includes every earlier record still connected to this group.
        if group_sites and pos > max_end:
            stats["overlap_groups"] += 1
            stats["largest_overlap_group"] = max(stats["largest_overlap_group"], group_sites)
            yield Span(chrom, first_pos, max_end, group_sites)
            group_sites = 0
        if group_sites == 0:
            first_pos, max_end = pos, end
        else:
            max_end = max(max_end, end)
        group_sites += 1

        if progress_every and record_number % progress_every == 0:
            print(
                f"Scanned {record_number:,} records; at {chrom}:{pos:,}; "
                f"{time.perf_counter() - started:,.1f} s elapsed.",
                flush=True,
            )

    if group_sites:
        stats["overlap_groups"] += 1
        stats["largest_overlap_group"] = max(stats["largest_overlap_group"], group_sites)
        yield Span(chrom, first_pos, max_end, group_sites)


def pack_groups(groups: Iterable[Span], target_sites: int):
    """Choose the nearest safe cut for each shard; prefer the smaller count on ties."""
    positive_integer(target_sites, "target_sites")
    packed = []
    pending = None

    for group in groups:
        if pending is not None and pending.sites < target_sites <= pending.sites + group.sites:
            below = target_sites - pending.sites
            above = pending.sites + group.sites - target_sites
            if below <= above:
                packed.append(pending)
                pending = None

        if pending is None:
            pending = group
        else:
            pending = Span(
                pending.chrom, pending.first_pos, group.end, pending.sites + group.sites
            )
        if pending.sites >= target_sites:
            packed.append(pending)
            pending = None

    if pending is not None:
        packed.append(pending)
    return packed


def shard_spans_from_lines(lines, target_sites, progress_every=0):
    """Pure streaming calculation, also used by the synthetic checks below."""
    positive_integer(target_sites, "target_sites")
    if isinstance(progress_every, bool) or not isinstance(progress_every, int) or progress_every < 0:
        raise ValueError("progress_every must be a nonnegative integer.")
    stats = {
        "chromosome": None,
        "records": 0,
        "maximum_end": 0,
        "overlap_groups": 0,
        "largest_overlap_group": 0,
    }
    spans = pack_groups(overlap_groups(lines, stats, progress_every), target_sites)
    if sum(span.sites for span in spans) != stats["records"]:
        raise RuntimeError("Internal error: shard counts do not sum to the input record count.")
    return spans, stats


def table_from_spans(spans, joint_vcf, joint_vcf_idx):
    """Place each internal cut immediately before the next shard's first POS."""
    if not spans:
        raise ValueError("The pVCF has no records; no shard intervals can be calculated.")
    rows = []
    interval_start = 1
    for i, span in enumerate(spans):
        interval_end = spans[i + 1].first_pos - 1 if i + 1 < len(spans) else span.end
        if span.first_pos < interval_start or span.end > interval_end:
            raise RuntimeError("Internal error: a variant span crosses a shard boundary.")
        shard_id = f"{i:04d}"
        rows.append([
            shard_id,
            span.sites,
            f"TBD{shard_id}",
            os.fspath(joint_vcf),
            os.fspath(joint_vcf_idx),
            "TBD",
            f"short.shard-{shard_id}-{span.chrom}-{interval_start}-{interval_end}",
            f"{span.chrom}:{interval_start}-{interval_end}",
        ])
        interval_start = interval_end + 1
    return pd.DataFrame(rows, columns=COLUMNS)

## Read the pVCF and write the table

`bcftools query` streams only `CHROM`, `POS`, `REF`, and raw `INFO/END` into Python. No genotype columns or intermediate site-only VCF are written. The full chromosome is scanned once; compressed genotype bytes still have to be read and decompressed. The supplied index path is checked for existence and copied unchanged into `joint_vcf_idx`; this sequential scan does not consult the index.

The TSV is written only after the scan succeeds and the record counts reconcile. An existing output TSV is replaced on a successful rerun. A failed scan leaves the existing output intact.

In [ ]:
def diagnostic_tail(handle, limit=16_384):
    handle.seek(0, os.SEEK_END)
    handle.seek(max(0, handle.tell() - limit))
    return handle.read().decode("utf-8", errors="replace").strip()


@contextmanager
def query_site_lines(vcf_path, bcftools="bcftools"):
    """Stream only four site fields. Check exit status and clean up on interruption."""
    executable = shutil.which(os.fspath(bcftools))
    if executable is None:
        raise FileNotFoundError(f"Cannot find executable: {bcftools!r}.")
    command = [
        executable, "query", "--allow-undef-tags", "--format",
        r"%CHROM\t%POS\t%REF\t%INFO/END\n", os.fspath(vcf_path),
    ]
    # A temporary stderr file avoids pipe deadlock without retaining large logs in RAM.
    with tempfile.TemporaryFile(mode="w+b") as stderr_file:
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=stderr_file,
            text=True, encoding="utf-8", errors="strict",
        )
        try:
            yield process.stdout
            process.stdout.close()
            return_code = process.wait()
            diagnostic = diagnostic_tail(stderr_file)
            if return_code != 0:
                raise RuntimeError(
                    f"bcftools query failed (exit {return_code}).\n"
                    f"Command: {shlex.join(command)}\n{diagnostic}"
                )
            if diagnostic:
                warnings.warn(f"bcftools diagnostics:\n{diagnostic}", RuntimeWarning)
        finally:
            process.stdout.close()
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=3)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()


def local_input_path(value, name):
    original = os.fspath(value)
    if not original or "://" in original or any(c in original for c in "\t\r\n"):
        raise ValueError(f"{name} must be a local filesystem path without tabs or newlines.")
    path = Path(original).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"{name} does not exist or is not a regular file: {path}")
    return original, path


def write_tsv(table, output_path):
    """Replace the output only after the complete table has been written successfully."""
    output_path = Path(output_path).expanduser()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", newline="", dir=output_path.parent,
            prefix=f".{output_path.name}.", suffix=".tmp", delete=False,
        ) as handle:
            temporary_path = Path(handle.name)
            table.to_csv(handle, sep="\t", index=False, lineterminator="\n")
        os.replace(temporary_path, output_path)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


def build_shard_table(
    joint_vcf, joint_vcf_idx, target_sites, output_tsv,
    *, progress_every=1_000_000, bcftools="bcftools",
):
    """Scan a local, single-chromosome pVCF, write the TSV, and return its DataFrame."""
    positive_integer(target_sites, "target_sites")
    original_vcf, vcf_path = local_input_path(joint_vcf, "joint_vcf")
    original_index, index_path = local_input_path(joint_vcf_idx, "joint_vcf_idx")
    output_path = Path(output_tsv).expanduser().resolve()
    if output_path in {vcf_path, index_path}:
        raise ValueError("output_tsv must differ from the pVCF and its index.")
    if output_path.exists() and not output_path.is_file():
        raise ValueError(f"output_tsv is not a regular file: {output_path}")

    started = time.perf_counter()
    with query_site_lines(vcf_path, bcftools) as lines:
        spans, stats = shard_spans_from_lines(lines, target_sites, progress_every)
    table = table_from_spans(spans, original_vcf, original_index)
    write_tsv(table, output_path)
    stats.update({
        "target_sites": target_sites,
        "shards": len(table),
        "elapsed_seconds": time.perf_counter() - started,
        "output_tsv": str(output_path),
    })
    table.attrs["summary"] = stats
    return table

## Small synthetic check

This example includes records sharing a POS, a deletion represented by REF length, and a chain spanning positions 150–200 through `INFO/END`. At a target of 5 records, it produces counts of 4, 5, and 4. The example paths below are placeholders.

In [ ]:
from IPython.display import display

# Synthetic records only. This cell does not read or write your pVCF.
demo_records = [
    (100, "A", "."), (100, "A", "."),
    (110, "ACGT", "."), (112, "A", "."),
    (150, "A", "180"), (160, "A", "200"),
    (190, "A", "."), (200, "A", "."),
    (250, "A", "."),
    (300, "AT", "."), (301, "A", "."),
    (400, "A", "."), (500, "AT", "."),
]
demo_lines = (f"chr6\t{pos}\t{ref}\t{end}\n" for pos, ref, end in demo_records)
demo_spans, demo_stats = shard_spans_from_lines(demo_lines, target_sites=5)
demo_df = table_from_spans(demo_spans, "/example/chr6.vcf.gz", "/example/chr6.vcf.gz.tbi")
assert demo_df["expected_number_of_sites"].tolist() == [4, 5, 4]
assert demo_df["region"].tolist() == ["chr6:1-149", "chr6:150-299", "chr6:300-501"]
assert demo_stats["records"] == 13
print("Synthetic check passed: target 5 records; shard counts 4, 5, 4.")
display(demo_df.head())

## Generate your TSV

The next cell reads the configured pVCF, writes the table, and displays its head. Use `df` for further inspection. Shard IDs remain strings such as `0000`. If you later read the TSV with pandas, use `dtype={"entity:split_short_regions_id": str}` to preserve the leading zeros.

In [ ]:
df = build_shard_table(
    JOINT_VCF,
    JOINT_VCF_IDX,
    TARGET_SITES,
    OUTPUT_TSV,
    progress_every=PROGRESS_EVERY,
    bcftools=BCFTOOLS,
)

summary = df.attrs["summary"]
counts = df["expected_number_of_sites"]
print(f"Wrote {summary['output_tsv']}")
print(
    f"{summary['chromosome']}: {summary['records']:,} records in {summary['shards']:,} shards; "
    f"target {TARGET_SITES:,}; {summary['elapsed_seconds']:,.1f} s elapsed."
)
print(
    f"Record counts: min {counts.min():,}, median {counts.median():,.0f}, "
    f"max {counts.max():,}; final shard {counts.iloc[-1]:,}."
)
print(f"Largest indivisible overlap group: {summary['largest_overlap_group']:,} records.")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 100):
    display(df.head(HEAD_ROWS))

## Reference

The query format and missing-tag behavior follow the [BCFtools manual, `query`](https://samtools.github.io/bcftools/bcftools.html#query). `--allow-undef-tags` emits `.` if the `END` tag is absent from the header, allowing the REF-length fallback. The boundary and packing rules implement the agreed specification above.